# Content assertions and successors

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import the exact published
verifier **2.0.0**.

Verification returns facts; the host still owns trusted inputs, authorization,
replay and revocation. Only safe semantic results are displayed; generated keys,
signatures and nonces stay in memory.

## Load the published verifier

Check the package identity and the portable backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import {
  BACKEND_ID,
  boundsNew,
  contentAssertion,
  strUtf8,
} from "npm:@bounded-authority-protocol/verifier@2.0.0";
import {
  createHash,
  generateKeyPairSync,
  type KeyObject,
  sign,
} from "node:crypto";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with {
  type: "json",
};
import assert from "node:assert/strict";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(
  JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }),
);

{"version":"2.0.0","backend":"portable"}


## Prepare the publisher

The helper hashes content, signs the standard JWS input and verifies against
caller-supplied context before returning facts.

In [ ]:
const publicKeyBytes = (key: KeyObject): Uint8Array =>
  Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const publisher = generateKeyPairSync("ed25519");
const attestor = {
  keyId: "publisher-1",
  publicKey: publicKeyBytes(publisher.publicKey),
  validFrom: 0,
  validBefore: null,
};
const profileDigest = createHash("sha256").update("retention-policy schema 3")
  .digest();
const bounds = boundsNew();

function publish(text: string, gen: number, prev: Uint8Array, iat: number) {
  const contentDigest = contentAssertion.contentDigest(strUtf8(text));
  assert.ok(contentDigest.ok);
  const input = contentAssertion.assertionSigningInput({
    attestorKeyId: "publisher-1",
    jti: `retention-policy-${gen}`,
    iss: "https://records.example",
    aud: "https://archive.example",
    sub: "policy:retention",
    profile: "urn:example:retention:3",
    profileDigest,
    contentDigest: contentDigest.value,
    gen,
    prev,
    iat,
    nbf: iat,
    exp: iat + 86_400,
  });
  assert.ok(input.ok);
  const signingBytes = Buffer.concat([
    input.value.protectedSegment,
    Buffer.from("."),
    input.value.payloadSegment,
  ]);
  const compact = contentAssertion.assembleCompact(
    input.value,
    sign(null, signingBytes, publisher.privateKey),
  );
  assert.ok(compact.ok);
  const facts = contentAssertion.verifyAssertion(compact.value, {
    attestor,
    issuer: "https://records.example",
    audience: "https://archive.example",
    subject: "policy:retention",
    profile: "urn:example:retention:3",
    profileDigest,
    contentDigest: contentDigest.value,
    now: iat + 10,
    bounds,
  });
  assert.ok(facts.ok);
  return facts.value;
}

console.log(
  JSON.stringify({
    publicKeyBytes: attestor.publicKey.length,
    profileDigestBytes: profileDigest.length,
  }),
);

{"publicKeyBytes":32,"profileDigestBytes":32}


## Verify the first assertion

Start at generation one with the genesis predecessor digest.


In [ ]:
const first = publish("keep records 7 years", 1, new Uint8Array(32), 1_000);
assert.equal(first.gen, 1);
console.log(JSON.stringify({ generation: first.gen }));


{"generation":1}


## Verify the next generation

A successor names the previous digest and increments its generation by one.


In [ ]:
const second = publish("keep records 10 years", 2, first.digest, 2_000);
const successor = contentAssertion.verifySuccessor(first, second, bounds);
assert.ok(successor.ok);
console.log(JSON.stringify({ ok: successor.ok, generation: second.gen }));


{"ok":true,"generation":2}


## Refuse a skipped generation

A valid signature does not make a skipped generation a successor.


In [ ]:
const third = publish("keep records 5 years", 3, second.digest, 3_000);
const skipped = contentAssertion.verifySuccessor(first, third, bounds);
assert.deepEqual(skipped, { ok: false });
console.log(JSON.stringify(skipped));


{"ok":false}


## Refuse a fork

The next generation must name the actual predecessor digest.


In [ ]:
const fork = publish(
  "keep records 1 year",
  2,
  new Uint8Array(32).fill(9),
  2_500,
);
const forked = contentAssertion.verifySuccessor(first, fork, bounds);
assert.deepEqual(forked, { ok: false });
console.log(JSON.stringify(forked));

{"ok":false}


## Finish

No private material was written to disk. Shut down the kernel to discard the
demonstration keys. Continue with the
[getting started guide](../guides/getting-started.md).